## <center>CSE 546: Reinforcement Learning</center>
### <center>Prof. Alina Vereshchaka</center>
#### <center>Spring 2026</center>

Welcome to the Assignment 3, Part 1: Introduction to Actor-Critic Methods! It includes the implementation of simple actor and critic networks and best practices used in modern Actor-Critic algorithms.

## Section 0: Setup and Imports

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## Section 1: Actor-Critic Network Architectures and Loss Computation

In this section, you will explore two common architectural designs for Actor-Critic methods and implement their corresponding loss functions using dummy tensors. These architectures are:
- A. Completely separate actor and critic networks
- B. A shared network with two output heads

Both designs are widely used in practice. Shared networks are often more efficient and generalize better, while separate networks offer more control and flexibility.

---


### Task 1a – Separate Actor and Critic Networks with Loss Function

Define a class `SeparateActorCritic`. Your goal is to:
- Create two completely independent neural networks: one for the actor and one for the critic.
- The actor should output a probability distribution over discrete actions (use `nn.Softmax`).
- The critic should output a single scalar value.

 Use `nn.ReLU()` as your activation function. Include at least one hidden layer of reasonable width (e.g. 64 or 128 units).

```python
# TODO: Define SeparateActorCritic class
```

 Next, simulate training using dummy tensors:
1. Generate dummy tensors for log-probabilities, returns, estimated values, and entropies.
2. Compute the actor loss using the advantage (return - value).
3. Compute the critic loss as mean squared error between values and returns.
4. Use a single optimizer for both the Actor and the Critic. In this case, combine the actor and critic losses into a total loss and perform backpropagation.
5. Use a separate optimizers for both the Actor and the Critic. In this case, keep the actor and critic losses separate and perform backpropagation.

```python
# TODO: Simulate loss computation and backpropagation
```

🔗 Helpful references:
- PyTorch Softmax: https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html
- PyTorch MSE Loss: https://pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html

---

In [3]:
# TODO: Define a class SeparateActorCritic with separate networks for actor and critic

# BEGIN_YOUR_CODE
class SeparateActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_actions):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,num_actions),
            nn.Softmax(dim=-1)
        )

        self.critic = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

input_dim = 4
hidden_dim = 64
num_actions = 4

model = SeparateActorCritic(input_dim, hidden_dim, num_actions)
  
batch_size =4
dummy_obs = torch.randn(batch_size,input_dim)
dummy_returns = torch.randn(batch_size,1)
dummy_logProbs = torch.randn(batch_size,requires_grad=True)
dummy_values = torch.randn(batch_size,1,requires_grad=True)

advantage = (dummy_returns - dummy_values).squeeze()

actor_loss = -(dummy_logProbs*advantage).mean()
critic_loss = F.mse_loss(dummy_values,dummy_returns)

actor_params = list(model.actor.parameters())
critic_params = list(model.critic.parameters())

all_params = actor_params + critic_params

optimizer = optim.Adam(all_params, lr = 1e-3)

optimizer.zero_grad()

total_loss = actor_loss + critic_loss
total_loss.backward()

optimizer.step()

dummy_obs_2 = torch.randn(batch_size,input_dim)
dummy_returns_2 = torch.randn(batch_size,1)
dummy_logProbs_2 = torch.randn(batch_size,requires_grad=True)
dummy_values_2 = torch.randn(batch_size,1,requires_grad=True)

advantage_2 = (dummy_returns_2 - dummy_values_2).squeeze()

actor_loss_2 = -(dummy_logProbs_2*advantage_2).mean()
critic_loss_2 = F.mse_loss(dummy_values_2,dummy_returns_2)

actor_optimizer = optim.Adam(model.actor.parameters(), lr = 1e-3)
actor_optimizer.zero_grad()
actor_loss_2.backward()
actor_optimizer.step()

critic_optimizer = optim.Adam(model.critic.parameters(), lr = 1e-3)
critic_optimizer.zero_grad()
critic_loss_2.backward()
critic_optimizer.step()
# END_YOUR_CODE

### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:
Having seperate actor and critic networks lets them be completeely independent and works well when they are learning different things, like when actor is learning actions and critic is learning values. The con of doing it this way that is its very expecsive because now youre maintaining two full networks.

### Task 1b – Shared Network with Actor and Critic Heads + Loss Function

Now define a class `SharedActorCritic`:
- Build a shared base network (e.g., linear layer + ReLU)
- Create two heads: one for actor (output action probabilities) and one for critic (output state value)

```python
# TODO: Define SharedActorCritic class
```

Then:
1. Pass a dummy input tensor through the model to obtain action probabilities and value.
2. Simulate dummy rewards and compute advantage.
3. Compute the actor and critic losses, combine them, and backpropagate.

```python
# TODO: Simulate shared network loss computation and backpropagation
```

 Use `nn.Softmax` for actor output and `nn.Linear` for scalar critic output.

🔗 More reading:
- Policy Gradient Methods: https://spinningup.openai.com/en/latest/algorithms/vpg.html
- Actor-Critic Overview: https://www.tensorflow.org/agents/tutorials/6_reinforce_tutorial
- PyTorch Categorical Distribution: https://pytorch.org/docs/stable/distributions.html#categorical

---

In [4]:
# BEGIN_YOUR_CODE
class SharedActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_actions):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )

        self.actorHead = nn.Sequential(
            nn.Linear(hidden_dim,num_actions),
            nn.Softmax(dim=-1)
        )

        self.criticHead = nn.Sequential(
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self,x):
        shared_output = self.shared(x)
        actorHead_output = self.actorHead(shared_output)
        criticHead_output = self.criticHead(shared_output)

        return actorHead_output, criticHead_output
    
    
model_shared = SharedActorCritic(4,64,4)

action_probs, value = model_shared(dummy_obs)

advantage = (dummy_returns - value).squeeze()

log_probs = torch.log(action_probs[:, 0])

actor_loss = -(log_probs*advantage).mean()
critic_loss = F.mse_loss(value,dummy_returns)

optimizer_shared = optim.Adam(model_shared.parameters(), lr = 1e-3)

optimizer_shared.zero_grad()

total_loss = actor_loss + critic_loss
total_loss.backward()

optimizer_shared.step()


# END_YOUR_CODE

### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

Shared networks work great when the actor and critic ar learning / trying to solve related problems. Having a shared network can help them learn a common representation. Also havig a shared netwrok means learning feweer total paramaters which can be computationally efficient. 

## Section 2: Auto-Adaptive Network Setup for Environments

You will now create a function that builds a shared actor-critic network that adapts to any Gymnasium environment. This function should inspect the environment and build input/output layers accordingly.

### Task 2: Auto-generate Input and Output Layers
Write a function `create_shared_network(env)` that constructs a neural network using the following rules:
- The input layer should match the environment's observation space.
- The output layer for the **actor** should depend on the action space:
  - For discrete actions: output probabilities using `nn.Softmax`.
  - For continuous actions: output mean and log std for a Gaussian distribution.
- The **critic** always outputs a single scalar value.

```python
# TODO: Define function `create_shared_network(env)`
```

#### Environments to Support:
Test your function with the following environments:
1. `CliffWalking-v0` (Use one-hot encoding for discrete integer observations.)
2. `LunarLander-v3` (Standard Box space for observations and discrete actions.)
3. `PongNoFrameskip-v4` (Use gym wrappers for Atari image preprocessing.)
4. `HalfCheetah-v5` (Continuous observation and continuous action.)

```python
# TODO: Loop through environments and test `create_shared_network`
```

Hint: Use `gym.spaces` utilities to determine observation/action types dynamically.

🔗 Observation/Action Space Docs:
- https://gymnasium.farama.org/api/spaces/

---

In [5]:
%pip install "gymnasium[mujoco]"
%pip install "gymnasium[atari]" "autorom[accept-rom-license]"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
class SharedActorCriticContinuous(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_actions):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )

        self.actorMean = nn.Sequential(
            nn.Linear(hidden_dim,num_actions)
        )

        self.actorLog = nn.Sequential(
            nn.Linear(hidden_dim,num_actions)
        )

        self.criticHead = nn.Sequential(
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self,x):
        shared_output = self.shared(x)
        mean = self.actorMean(shared_output)
        log = self.actorLog(shared_output)
        criticHead_output = self.criticHead(shared_output)

        return mean,log , criticHead_output
    

In [10]:
!python3 -m autorom.accept_rom_license
import ale_py
gym.register_envs(ale_py)

/Library/Developer/CommandLineTools/usr/bin/python3: Error while finding module specification for 'autorom.accept_rom_license' (ModuleNotFoundError: No module named 'autorom')


In [11]:
import gymnasium as gym
envs = gym.pprint_registry()

===== classic_control =====
Acrobot-v1                  CartPole-v0                 CartPole-v1
MountainCar-v0              MountainCarContinuous-v0    Pendulum-v1
===== phys2d =====
phys2d/CartPole-v0          phys2d/CartPole-v1          phys2d/Pendulum-v0
===== box2d =====
BipedalWalker-v3            BipedalWalkerHardcore-v3    CarRacing-v3
LunarLander-v3              LunarLanderContinuous-v3
===== toy_text =====
Blackjack-v1                CliffWalking-v1             CliffWalkingSlippery-v1
FrozenLake-v1               FrozenLake8x8-v1            Taxi-v3
===== tabular =====
tabular/Blackjack-v0        tabular/CliffWalking-v0
===== None =====
Ant-v2                      Ant-v3                      GymV21Environment-v0
GymV26Environment-v0        HalfCheetah-v2              HalfCheetah-v3
Hopper-v2                   Hopper-v3                   Humanoid-v2
Humanoid-v3                 HumanoidStandup-v2          InvertedDoublePendulum-v2
InvertedPendulum-v2         Pusher-v2             

In [14]:
class SharedActorCriticCNN(nn.Module):
    def __init__(self, hidden_dim, num_actions):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(4,32,kernel_size=8,stride=4),
            nn.ReLU(),
            nn.Conv2d(32,64,kernel_size=4,stride=2),
            nn.ReLU(),
            nn.Conv2d(64,64,kernel_size=3,stride=1),
            nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, hidden_dim),
            nn.ReLU()
        )
       
        self.actorHead = nn.Sequential(
            nn.Linear(hidden_dim,num_actions),
            nn.Softmax(dim=-1)
        )

        self.criticHead = nn.Sequential(
            nn.Linear(hidden_dim, 1)
        )
   
    def forward(self,x):
        conv_output = self.conv(x)
        fc = self.fc(conv_output)
        actor_head = self.actorHead(fc)
        critic_head = self.criticHead(fc)
      

        return actor_head, critic_head


In [16]:
# BEGIN_YOUR_CODE

def create_shared_network(env):
    if isinstance(env.observation_space, gym.spaces.Discrete):
        input_dim = env.observation_space.n

    elif isinstance(env.observation_space, gym.spaces.Box):
        if len(env.observation_space.shape) == 1:
            input_dim = env.observation_space.shape[0]
        elif len(env.observation_space.shape) > 1:
            num_actions = env.action_space.n
            return SharedActorCriticCNN(64,num_actions)

    if isinstance(env.action_space, gym.spaces.Discrete):
        num_actions = env.action_space.n
    elif isinstance(env.action_space, gym.spaces.Box):
        num_actions = env.action_space.shape[0]
        return SharedActorCriticContinuous(input_dim, 64, num_actions)
    return SharedActorCritic(input_dim,64, num_actions)

env = gym.make('PongNoFrameskip-v4')
create_shared_network(env)

# END_YOUR_CODE

SharedActorCriticCNN(
  (conv): Sequential(
    (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=64, bias=True)
    (2): ReLU()
  )
  (actorHead): Sequential(
    (0): Linear(in_features=64, out_features=6, bias=True)
    (1): Softmax(dim=-1)
  )
  (criticHead): Sequential(
    (0): Linear(in_features=64, out_features=1, bias=True)
  )
)

### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

Instead of manually creting a new network for each type of env, using creat_shared_netwrok() allows you have have reuable code as it inspects the env and decides how the network should be designed.

### Task 3: Write Observation Normalization Function
Create a function `normalize_observation(obs, env)` that:
- Checks if the observation space is `Box` and has `low` and `high` attributes.
- If so, normalize the input observation.
- Otherwise, return the observation unchanged.

```python
# TODO: Define `normalize_observation(obs, env)`
```

Test this function with observations from:
- `LunarLander-v3`
- `PongNoFrameskip-v4`

Note: Atari observations are image arrays. Normalize pixel values to [0, 1]. For LunarLander-v3, the different elements in the observation vector have different ranges. Normalize them to [0, 1] using the `low` and `high` attributes of the observation space.


---

In [23]:
# BEGIN_YOUR_CODE

def normalize_observation(obs,env):
    
    if isinstance(env.observation_space, gym.spaces.Box):
        if len(env.observation_space.shape) == 1:
            low = env.observation_space.low
            high = env.observation_space.high
            if np.all(np.isfinite(low)) and np.all(np.isfinite(high)):
                normalized = 2 * (obs - low) / (high - low) - 1
                return normalized
            else:
                return obs
        elif len(env.observation_space.shape) > 1:
            normalized = obs / 255.0
            return normalized
    else:
        return obs

env = gym.make('PongNoFrameskip-v4')
obs, _ = env.reset()

result = normalize_observation(obs,env)
print(result)


# END_YOUR_CODE

[[[0.         0.         0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]
  ...
  [0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]]

 [[0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]
  ...
  [0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]]

 [[0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]
  ...
  [0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]
  [0.42745098 0.4627451  0.16862745]]

 ...

 [[0.20784314 0.37254902 0.09411765]
  [0.20784314 0.37254902 0.09411765]
  [0.20784314 0.37254902 0.09411765]
  ...
  [0.20784314 0.37254902 0.09411765]
  [0.20784314 0.37254902 0.09411765]
  [0.20784314 0.37254902 0.09411765]]

 [[0.20784314 0.37254902 0.09411765]
  [0.20784314 0.37254902 0.09411765]


### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

when we get observations from different environments, the numbers can range from very very small to pretty large , and in cases like these, the very large numbers can dominate and make the training unstable.
this is why it is important to normalize.

## Section 4: Gradient Clipping

To prevent exploding gradients, it's common practice to clip gradients before optimizer updates.

### Task 4: Clip Gradients for Actor-Critic Networks
Use dummy tensors and apply gradient clipping with the following PyTorch method:
```python
# During training, after loss.backward():
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
```

Reuse the loss computation from Task 1a or 1b. After computing the gradients, apply gradient clipping.
Print the gradient norm before and after clipping to verify it’s applied.

🔗 PyTorch Docs: https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html


---

In [24]:
# BEGIN_YOUR_CODE

model_shared = SharedActorCritic(4,64,4)

action_probs, value = model_shared(dummy_obs)

advantage = (dummy_returns - value).squeeze()

log_probs = torch.log(action_probs[:, 0])

actor_loss = -(log_probs*advantage).mean()
critic_loss = F.mse_loss(value,dummy_returns)

optimizer_shared = optim.Adam(model_shared.parameters(), lr = 1e-3)

optimizer_shared.zero_grad()

total_loss = actor_loss + critic_loss
total_loss.backward()

total_norm = 0
for p in model_shared.parameters():
    if p.grad is not None:
        total_norm += p.grad.norm().item() ** 2
total_norm = total_norm ** 0.5
print(f"Gradient norm before clipping: {total_norm}")

torch.nn.utils.clip_grad_norm_(model_shared.parameters(), max_norm=0.5)

total_norm = 0
for p in model_shared.parameters():
    if p.grad is not None:
        total_norm += p.grad.norm().item() ** 2
total_norm = total_norm ** 0.5
print(f"Gradient norm after clipping: {total_norm}")

optimizer_shared.step()
# END_YOUR_CODE

Gradient norm before clipping: 3.125086981528342
Gradient norm after clipping: 0.4999999164984485


### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

Exploding gradients, sometimes during backprop, the gradients become very large and the network weights get updated by a hige amount casuing the training to destabilize. This is why we clip the gradient.

If you are working in a team, provide a contribution summary.
| Team Member | Step# | Contribution (%) |
|---|---|---|
|   | Task 1 |   |
|   | Task 2 |   |
|   | Task 3 |   |
|   | Task 4 |   |
|   | **Total** |   |
